# `music.mid` と `drums.mid` を結合

MuScripter担当の非ドラムMIDIと、DrumScripter担当のドラムMIDIを `complete.mid` にまとめます。学習・推論はせず、楽器トラックを維持して結合します。**テンポ設定は `music.mid` を採用**し、音符の時刻は秒単位で維持します。両者の音声開始位置が異なる場合はドラム側のオフセットを指定してください。

In [ ]:
%pip -q install "pretty_midi>=0.2.10,<1"


## 非ドラムMIDIをアップロード

In [ ]:
from google.colab import files
import io
import pretty_midi

uploaded_music = files.upload()
if len(uploaded_music) != 1 or not next(iter(uploaded_music)).lower().endswith(('.mid', '.midi')):
    raise ValueError('MIDIファイルを1つ選んでください。')
music = pretty_midi.PrettyMIDI(io.BytesIO(next(iter(uploaded_music.values()))))
music.instruments = [i for i in music.instruments if not i.is_drum]
if not any(i.notes for i in music.instruments):
    raise ValueError('非ドラムの音符が含まれていません。')
print('非ドラム:', [(i.name, len(i.notes)) for i in music.instruments])


## ドラムMIDIをアップロード

In [ ]:
uploaded_drums = files.upload()
if len(uploaded_drums) != 1 or not next(iter(uploaded_drums)).lower().endswith(('.mid', '.midi')):
    raise ValueError('ドラムMIDIを1つ選んでください。')
drums = pretty_midi.PrettyMIDI(io.BytesIO(next(iter(uploaded_drums.values()))))
drum_tracks = [i for i in drums.instruments if i.is_drum]
if not any(i.notes for i in drum_tracks):
    raise ValueError('ドラムトラック（MIDIチャンネル10）が見つかりません。')
print('ドラム:', [(i.name, len(i.notes)) for i in drum_tracks])


## 時刻を合わせて結合

`DRUM_OFFSET_SECONDS` はドラムを後ろへ移動する秒数です。初期値0なら元の時間位置を保持します。元ファイルは変更しません。

In [ ]:
import copy
from pathlib import Path
import tempfile

DRUM_OFFSET_SECONDS = 0.0
if not isinstance(DRUM_OFFSET_SECONDS, (int, float)) or not 0 <= DRUM_OFFSET_SECONDS <= 3600:
    raise ValueError('DRUM_OFFSET_SECONDS は 0〜3600 秒の数値にしてください。')

complete = copy.deepcopy(music)  # music のテンポ変化・拍子等を保持
for source in drum_tracks:
    instrument = copy.deepcopy(source)
    for note in instrument.notes:
        note.start += DRUM_OFFSET_SECONDS
        note.end += DRUM_OFFSET_SECONDS
    for cc in instrument.control_changes:
        cc.time += DRUM_OFFSET_SECONDS
    for bend in instrument.pitch_bends:
        bend.time += DRUM_OFFSET_SECONDS
    complete.instruments.append(instrument)

work_dir = Path(tempfile.mkdtemp(prefix='midi_merge_', dir='/content'))
output_path = work_dir / 'complete.mid'
complete.write(str(output_path))
print('結合後:', len(complete.instruments), 'トラック /',
      sum(len(i.notes) for i in complete.instruments), '音符')
files.download(str(output_path))
